k1 is min entropy of the bitstring

In [24]:
import cryptomite as cm
import numpy as np
import os

In [25]:
file_paths = {
    "12M": "../q_gen_input/12M_quantum.bin",
    "10k": "../q_gen_input/10k_quantum.bin",
    "1024": "../q_gen_input/1024_quantum.bin",
    "10k_0s": "../q_gen_input/10k_0s.bin",
    "420k": "../q_gen_input/420k_q_error_corrected.bin"
}

methods = ["toeplitz", "circulant", "dodis"]#, "trevisan"


In [26]:
def estimate_min_entropy_frequency(bit_array):
    n = len(bit_array)
    counts = np.bincount(bit_array)
    if len(counts) < 2 or np.sum(counts) == 0:
        return 0.0, 0.0
    p_max = np.max(counts) / n
    per_bit_entropy = -np.log2(p_max)
    total_min_entropy = n * per_bit_entropy
    return total_min_entropy, per_bit_entropy

In [27]:
def load_binary(file_path):
    with open(file_path, "rb") as f:
        data = f.read()
    bit_array = np.unpackbits(np.frombuffer(data, dtype=np.uint8))
    return bit_array


In [28]:
data_sets = {}
for name, path in file_paths.items():
    if os.path.exists(path):
        data_sets[name] = load_binary(path)
        print(f"Loaded {name}: {len(data_sets[name])} bits")
    else:
        print(f"File {path} not found.")


Loaded 12M: 12700000 bits
Loaded 10k: 10000 bits
Loaded 1024: 1024 bits
Loaded 10k_0s: 10000 bits
Loaded 420k: 420000 bits


In [29]:
k1_values = {}
per_bit_entropies = {}
for name, bit_array in data_sets.items():
    k1_values[name], per_bit_entropies[name] = estimate_min_entropy_frequency(bit_array)
    print(k1_values[name])

12272957.200284077
9902.2287838847
1009.643038397507
60.7207948340012
405322.3411238289


In [30]:
def extract_randomness_toeplitz(bit_array, k1, epsilon=2**-32):
    n = len(bit_array)
    m = max(1, int(k1 - 2 * np.log2(1 / epsilon)))
    seed_length = n + m - 1
    extractor = cm.toeplitz.Toeplitz(n, m)
    seed = np.random.randint(0, 2, seed_length).tolist()
    extracted_bits = extractor.extract(bit_array.tolist(), seed)
    return extracted_bits
    

In [31]:
def extract_randomness_circulant(bit_array, k1, epsilon=2**-32):
    n = len(bit_array)
    # Ensure that n+1 is prime:
    required_seed_length = cm.utils.previous_prime(n + 1)
    n_valid = required_seed_length - 1
    if n_valid != n:
        print(f"Warning: For circulant extraction, adjusting input length from {n} to {n_valid} to meet prime requirements.")
        bit_array = bit_array[:n_valid]
        n = n_valid
    m = max(1, int(k1 - 2 * np.log2(1 / epsilon)))
    seed_length = n + 1  # Now guaranteed to be prime.
    extractor = cm.circulant.Circulant(n, m)
    seed = np.random.randint(0, 2, seed_length).tolist()
    extracted_bits = extractor.extract(bit_array.tolist(), seed)
    return extracted_bits


In [32]:
def extract_randomness_dodis(bit_array, k1, k2=None, epsilon=2**-32):
    n = len(bit_array)
    n_valid = cm.utils.previous_na_set(n)
    if n_valid != n:
        print(f"Warning: For Dodis extraction, adjusting input length from {n} to {n_valid} to meet prime with primitive root 2 requirement.")
        bit_array = bit_array[:n_valid]
        n = n_valid
    
    if k2 is None:
        k2 = n
    
    m = max(1, int(k1 + k2 - n - 2 * np.log2(1 / epsilon)))
    
    seed_bits = np.random.randint(0, 2, n).tolist()
    
    extractor = cm.dodis.Dodis(n, m)
    
    extracted_bits = extractor.extract(list(bit_array), seed_bits)
    return extracted_bits


In [33]:
def extract_randomness_trevisan(bit_array, k1, error=2**-32):
    n = len(bit_array)
    # Create the extractor; note that k1 is cast to int.
    extractor = cm.trevisan.Trevisan(n, int(k1), error)
    # Compute seed length as a sum of two logarithms:
    seed_length = int(np.ceil(np.log2(n))) + int(np.ceil(np.log2(1 / error)))
    # Generate a seed of that length.
    seed = np.random.randint(0, 2, seed_length).tolist()
    extracted_bits = extractor.extract(bit_array.tolist(), seed)
    return extracted_bits

In [34]:
extracted_data = {}
for name, bits in data_sets.items():
    print(f"extracting from {name}")
    if len(bits) > 1 and name in k1_values:
        k1 = k1_values[name]
        # Ensure there are enough bits for extraction
        if k1 > 0:
            extracted_data[name] = {
                "toeplitz": extract_randomness_toeplitz(bits, k1),
                "circulant": extract_randomness_circulant(bits, k1),
                "dodis": extract_randomness_dodis(bits, k1)
                #"trevisan": extract_randomness_trevisan(bits, k1)
            }
            print(f"Extracted random bits for {name}")
        else:
            print(f"Skipping {name}, insufficient min-entropy (k1 <= 0)")
    else:
        print(f"Skipping {name}, not enough bits for extraction or missing k1 value")


extracting from 12M
Extracted random bits for 12M
extracting from 10k
Extracted random bits for 10k
extracting from 1024
Extracted random bits for 1024
extracting from 10k_0s
Extracted random bits for 10k_0s
extracting from 420k
Extracted random bits for 420k


In [35]:
entropies = {}
for name, methods in extracted_data.items():
    og_bit_array = data_sets[name]
    og_entropy = k1_values[name]
    og_entropy_per_bit = per_bit_entropies[name]
    entropies_name = {}
    for method, result in methods.items():
        method_entropy, method_entropy_per_bit = estimate_min_entropy_frequency(result)
        entropies_name[method] = {'overall_entropy' : method_entropy,
                                  'per_bit_entropy' : method_entropy_per_bit}
    entropies[name]=entropies_name
for name in file_paths.keys():
    print(f"For {name}, starting with a total entropy of {k1_values[name]} and a per bit entropy of {og_entropy_per_bit}")
    for method in methods:
        print(f"using {method} extraction")
        print(f"total entropy - {entropies[name][method]['overall_entropy']}")
        print(f"entropy per bit - {entropies[name][method]['per_bit_entropy']}")
    print("")

For 12M, starting with a total entropy of 12272957.200284077 and a per bit entropy of 0.9650531931519736
using toeplitz extraction
total entropy - 12267649.579989223
entropy per bit - 0.9995727641387587
using circulant extraction
total entropy - 12270603.590987934
entropy per bit - 0.9998134580809865
using dodis extraction
total entropy - 12272248.127060346
entropy per bit - 0.999947455507055

For 10k, starting with a total entropy of 9902.2287838847 and a per bit entropy of 0.9650531931519736
using toeplitz extraction
total entropy - 9697.315434225007
entropy per bit - 0.9856998815028468
using circulant extraction
total entropy - 9691.602721806461
entropy per bit - 0.9851192032736797
using dodis extraction
total entropy - 9532.573228244215
entropy per bit - 0.9689543838426729

For 1024, starting with a total entropy of 1009.643038397507 and a per bit entropy of 0.9650531931519736
using toeplitz extraction
total entropy - 926.3727963367805
entropy per bit - 0.9802886733722546
using cir

In [36]:
# for name, methods in extracted_data.items():
#     for method, bits in methods.items():
#         output_path = f"{name}_{method}_extracted.bin"
#         with open(output_path, "wb") as f:
#             f.write(np.packbits(bits))
#         print(f"Saved extracted random bits using {method} to {output_path}")
